# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedali287/flyrank-intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [42]:
!pip -q install datasets duckdb pandas pyarrow huggingface_hub

In [43]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [44]:
from huggingface_hub import login

login(token=HF_TOKEN)

In [45]:
from datasets import get_dataset_config_names

configs = get_dataset_config_names(
    "FlyRank/internship-warehouse",
    token=HF_TOKEN
)

configs


['dim_clients',
 'dim_content',
 'fact_content_daily_performance',
 'fact_content_query_90d']

In [46]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

In [47]:
dataset

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})

In [48]:
df.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events']

In [49]:
sample = dataset["train"].select(range(100000))

march_sample = sample.filter(
    lambda x: x["report_date"].year == 2026
    and x["report_date"].month == 3
)

In [50]:
sample = dataset["train"].select(range(100000))

In [51]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})


In [52]:
sample_data = dataset["train"].select(range(100000))

march_data = sample_data.filter(
    lambda x: x["report_date"].year == 2026
    and x["report_date"].month == 3
)

In [53]:
march_df = march_data.select(
    range(min(10000, len(march_data)))
).to_pandas()

march_df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [54]:
## Unit of Analysis + Time Window

#One row represents the daily performance of one content item for one client on one report date.

#The unit of analysis is:

#one content_hash_id × one client_hash_id × one report_date.

#I use March 2026 as my development window because it is a mid-panel month. I avoid the final month because it should remain a sealed test period.

In [55]:
march_df = march_data.select(
    range(min(10000, len(march_data)))
).to_pandas()

march_df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [56]:
## Field Classification

### Features

'''These fields are measurable before making a decision:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions


### Label / Proxy

This dataset does not directly contain a future decline label.

A possible future label would be a directional performance change measured after the prediction window.

Example:
future decline in clicks or sessions.


### Context

These fields are used for grouping and identification:

- client_hash_id
- content_hash_id
- report_date


### Excluded

The following are excluded:

- Any future outcome columns because they would leak information.
- Client/content IDs as model features because they identify records but do not represent useful patterns.
'''


'These fields are measurable before making a decision:\n\n- gsc_impressions\n- gsc_clicks\n- gsc_avg_position\n- ga4_pageviews\n- ga4_sessions\n\n\n### Label / Proxy\n\nThis dataset does not directly contain a future decline label.\n\nA possible future label would be a directional performance change measured after the prediction window.\n\nExample:\nfuture decline in clicks or sessions.\n\n\n### Context\n\nThese fields are used for grouping and identification:\n\n- client_hash_id\n- content_hash_id\n- report_date\n\n\n### Excluded\n\nThe following are excluded:\n\n- Any future outcome columns because they would leak information.\n- Client/content IDs as model features because they identify records but do not represent useful patterns.\n'

In [57]:
march_df.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events']

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [58]:
## Grain Check

#I verify that each combination of client, content, and date represents one row.

In [59]:
grain_check = (
    march_df
    .groupby(
        [
            "client_hash_id",
            "content_hash_id",
            "report_date"
        ]
    )
    .size()
    .reset_index(name="count")
)

grain_check[grain_check["count"] > 1].head()

,client_hash_id,content_hash_id,report_date,count


In [60]:
## Dataset Size and Date Window

#I check how many records exist in the selected window and confirm the dates.

In [61]:
print("Rows:", len(march_df))

print(
    "Start date:",
    march_df["report_date"].min()
)

print(
    "End date:",
    march_df["report_date"].max()
)

Rows: 0
Start date: nan
End date: nan


In [62]:
## Data Availability Check

#I filter using GA4 availability because missing GA4 history should not be treated as zero engagement.

In [63]:
available_rows = march_df[
    march_df["ga4_data_available"] == True
]

print(
    "Rows with GA4 available:",
    len(available_rows)
)

Rows with GA4 available: 0


In [64]:
## Missing Value Check

#I check missing values because missingness may follow patterns in the data and affect analysis.

In [65]:
missing_values = (
    march_df.isnull()
    .mean()
    .sort_values(ascending=False)
)

missing_values.head(10)

,0
report_date,NaN
client_hash_id,NaN
content_hash_id,NaN
client_has_gsc,NaN
client_has_ga4,NaN
gsc_data_available,NaN
ga4_data_available,NaN
gsc_impressions,NaN
gsc_clicks,NaN
gsc_sum_position,NaN


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [66]:
## Data Limitations

#This dataset only shows observed performance measurements and cannot explain the reasons behind user behavior.

#lient history is not equally available because different clients started collecting data at different times.

#ows without GSC or GA4 availability should not automatically be interpreted as zero performance.

#This data supports decision-support analysis, but it cannot prove that a recommendation will improve business results without an experiment.

## Self-check

Before you submit, confirm each line honestly:

- [T] Every section above is filled — markdown thinking AND the code that backs it
- [T] The notebook runs top to bottom with no errors (Runtime → Run all)
- [T] No client names, URLs, or private queries anywhere
- [T] My claims use careful words: observed, measured, directional, decision-support
- [T] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.